In [ ]:
# =====================================================================
# PHYSICAL AI DEMONSTRATION
#
# YOLO + REAL BOTTLE DETECTION + GEMINI + INTERACTIVE TARGET
# + ROBOT ANIMATION
#
# Google Colab
#
# Pipeline:
#
# Room Image
#    ↓
# YOLO detects actual bottle
#    ↓
# User selects delivery target
#    ↓
# Gemini creates high-level task plan
#    ↓
# Robot starts lower-left
#    ↓
# Robot moves to detected bottle
#    ↓
# Robot picks bottle
#    ↓
# Robot moves to selected target
#    ↓
# Robot releases bottle
#
# =====================================================================


# =====================================================================
# STEP 0
# INSTALL REQUIRED LIBRARIES
# =====================================================================

!pip -q install ultralytics google-genai pillow opencv-python-headless



In [ ]:

# =====================================================================
# STEP 1
# IMPORT LIBRARIES
# =====================================================================

import cv2
import base64
import numpy as np

from PIL import Image as PILImage
from PIL import ImageDraw

from IPython.display import display
from IPython.display import Image

from google.colab import files
from google.colab.output import eval_js
from google.colab import userdata

from google import genai

from ultralytics import YOLO

In [ ]:
# =====================================================================
# STEP 2
# GEMINI SETUP
#
# In Colab:
#
# Left sidebar
#      ↓
# Secrets
#      ↓
# Add:
#
# GEMINI_API_KEY
#
# =====================================================================

import os
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)
os.environ["GOOGLE_API_KEY"] = api_key
#import os
#from getpass import getpass

#os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API key: ")

In [ ]:
# ============================================================
# UPLOAD IMAGE + OBJECT DETECTION + BOUNDING BOXES + LABELS
# ============================================================

!pip install -q ultralytics

from google.colab import files
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Upload room image
# ------------------------------------------------------------

uploaded = files.upload()

image_file = list(uploaded.keys())[0]

# Load image
image = cv2.imread(image_file)

if image is None:
    raise Exception("Could not read the uploaded image.")

# ------------------------------------------------------------
# 2. Load YOLO
# ------------------------------------------------------------

model = YOLO("yolo11n.pt")

# ------------------------------------------------------------
# 3. Detect objects
# ------------------------------------------------------------

results = model(image, conf=0.20, verbose=False)

annotated_image = image.copy()

# ------------------------------------------------------------
# 4. Draw bounding boxes and labels
# ------------------------------------------------------------

for box in results[0].boxes:

    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

    class_id = int(box.cls[0])

    object_name = model.names[class_id]

    confidence = float(box.conf[0])

    label = f"{object_name} {confidence:.2f}"

    # Bounding box
    cv2.rectangle(
        annotated_image,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        3
    )

    # Label
    cv2.putText(
        annotated_image,
        label,
        (x1, max(y1 - 10, 25)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    print(
        f"{object_name:15s} "
        f"confidence={confidence:.2f} "
        f"box=({x1},{y1},{x2},{y2})"
    )

# ------------------------------------------------------------
# 5. Display annotated image
# ------------------------------------------------------------

plt.figure(figsize=(14, 9))

plt.imshow(
    cv2.cvtColor(
        annotated_image,
        cv2.COLOR_BGR2RGB
    )
)

plt.title("Robot Vision - Object Detection")
plt.axis("off")
plt.show()

In [ ]:
# =====================================================================
# STEP 3
# UPLOAD ROOM IMAGE
# =====================================================================

print(
    "\nPlease upload your ROOM IMAGE."
)

print(
    "The image should contain the plastic bottle on the table."
)


uploaded = files.upload()


image_path = next(
    iter(uploaded)
)


print(
    "\nUploaded:",
    image_path
)


In [ ]:
# =====================================================================
# STEP 4
# LOAD ROOM IMAGE
# =====================================================================

room_bgr = cv2.imread(
    image_path
)


if room_bgr is None:

    raise ValueError(
        "Could not read the uploaded image."
    )


room = cv2.cvtColor(

    room_bgr,

    cv2.COLOR_BGR2RGB

)


height, width, _ = room.shape


print(
    "\nImage dimensions:"
)

print(
    "Width:",
    width
)

print(
    "Height:",
    height
)


In [ ]:
# =====================================================================
# STEP 5
# LOAD YOLO
# =====================================================================

print(
    "\nLoading YOLO..."
)


yolo = YOLO(
    "yolo11n.pt"
)


print(
    "YOLO loaded."
)


# =====================================================================
# STEP 6
# DETECT OBJECTS
# =====================================================================

print(
    "\nYOLO is analysing the room..."
)


results = yolo(

    image_path,

    conf=0.20,

    verbose=False

)


result = results[0]


print(
    "\nObjects detected:"
)


detected_objects = []


for box in result.boxes:

    class_id = int(
        box.cls[0]
    )

    confidence = float(
        box.conf[0]
    )

    name = yolo.names[
        class_id
    ]


    coordinates = (

        box.xyxy[0]
        .cpu()
        .numpy()
    )


    x1, y1, x2, y2 = coordinates


    detected_objects.append(

        {

            "name": name,

            "confidence": confidence,

            "box": [
                float(x1),
                float(y1),
                float(x2),
                float(y2)
            ]

        }

    )


    print(

        f"{name:15s}"

        f" confidence = "

        f"{confidence:.2f}"

    )


In [ ]:
# =====================================================================
# STEP 7
# FIND ALL BOTTLES
# =====================================================================

bottles = [

    obj

    for obj in detected_objects

    if obj["name"] == "bottle"

]


print(
    "\nNumber of bottles detected:",
    len(bottles)
)


# =====================================================================
# STEP 8
# VERIFY BOTTLE DETECTION
# =====================================================================

if len(bottles) == 0:

    print(
        "\nYOLO DID NOT DETECT A BOTTLE."
    )

    print(
        "Try one of these:"
    )

    print(
        "1. Use a clearer image."
    )

    print(
        "2. Make sure the bottle is visible."
    )

    print(
        "3. Crop the image around the table."
    )

    print(
        "4. Try a larger YOLO model."
    )

    raise ValueError(
        "Bottle not detected."
    )



In [ ]:

# =====================================================================
# STEP 9
# SELECT THE BOTTLE
#
# If there is one bottle:
#     use it.
#
# If there are multiple bottles:
#     choose the bottle whose center is highest in the image.
#
# Why?
#
# In your room image, the bottle ON THE TABLE is normally higher
# than a bottle positioned on the floor.
#
# Smaller Y coordinate = higher position in image.
# =====================================================================


def bottle_center(
    bottle
):

    x1, y1, x2, y2 = bottle[
        "box"
    ]


    center_x = (
        x1 + x2
    ) / 2


    center_y = (
        y1 + y2
    ) / 2


    return np.array(

        [
            center_x,
            center_y
        ],

        dtype=float

    )


if len(bottles) == 1:

    selected_bottle = bottles[0]


else:

    selected_bottle = min(

        bottles,

        key=lambda b:
        bottle_center(b)[1]

    )


bottle_position = bottle_center(
    selected_bottle
)


bottle_box = selected_bottle[
    "box"
]


print(
    "\nSELECTED REAL BOTTLE"
)

print(
    "YOLO confidence:",
    round(
        selected_bottle[
            "confidence"
        ],
        3
    )
)

print(
    "Bottle center:",
    bottle_position.astype(int)
)


# =====================================================================
# STEP 10
# SHOW YOLO DETECTION BEFORE RUNNING ROBOT
# =====================================================================

preview = PILImage.fromarray(
    room.copy()
)


draw = ImageDraw.Draw(
    preview
)


x1, y1, x2, y2 = [

    int(v)

    for v in bottle_box

]


# Draw bounding box around actual bottle

draw.rectangle(

    [
        x1,
        y1,
        x2,
        y2
    ],

    outline="lime",

    width=6

)


draw.text(

    (
        x1,
        max(0, y1 - 25)
    ),

    "YOLO: REAL BOTTLE",

    fill="lime"

)


preview_file = (
    "/content/"
    "detected_bottle.jpg"
)


preview.save(
    preview_file
)


print(
    "\nYOLO selected this bottle:"
)


display(

    Image(
        filename=preview_file
    )

)


In [ ]:
# =====================================================================
# STEP 11
# USER SELECTS DELIVERY TARGET
# =====================================================================

print(
    "\nNow select where the robot should deliver the bottle."
)


_, encoded_buffer = cv2.imencode(

    ".jpg",

    room_bgr

)


image_base64 = base64.b64encode(

    encoded_buffer

).decode()


javascript = f"""

new Promise((resolve) => {{

    const container =
        document.createElement('div');


    container.innerHTML = `

        <h2>
        PHYSICAL AI DELIVERY TARGET
        </h2>

        <p>
        Click on the room image where
        the robot should deliver the bottle.
        </p>

        <canvas
        id="targetCanvas"
        style="
        border:3px solid black;
        cursor:crosshair;">
        </canvas>

        <p id="status">
        Waiting for target...
        </p>

    `;


    document.body.appendChild(
        container
    );


    const canvas =
        document.getElementById(
            "targetCanvas"
        );


    const ctx =
        canvas.getContext(
            "2d"
        );


    const img =
        new Image();


    img.src =
        "data:image/jpeg;base64,{image_base64}";


    img.onload = function() {{

        const maxWidth = 900;

        let scale = 1;


        if (
            img.width >
            maxWidth
        ) {{

            scale =
                maxWidth /
                img.width;

        }}


        canvas.width =
            img.width *
            scale;


        canvas.height =
            img.height *
            scale;


        ctx.drawImage(

            img,

            0,

            0,

            canvas.width,

            canvas.height

        );


        canvas.onclick =
        function(event) {{

            const rect =
                canvas.getBoundingClientRect();


            const clickX =
                event.clientX -
                rect.left;


            const clickY =
                event.clientY -
                rect.top;


            const originalX =
                clickX /
                scale;


            const originalY =
                clickY /
                scale;


            // Draw target circle

            ctx.beginPath();


            ctx.arc(

                clickX,

                clickY,

                22,

                0,

                2 * Math.PI

            );


            ctx.strokeStyle =
                "red";


            ctx.lineWidth =
                7;


            ctx.stroke();


            ctx.font =
                "22px Arial";


            ctx.fillStyle =
                "red";


            ctx.fillText(

                "TARGET",

                clickX + 28,

                clickY

            );


            document.getElementById(
                "status"
            ).innerHTML =
                "Delivery target selected.";


            resolve([

                originalX,

                originalY

            ]);

        }};

    }};

}})

"""


target_coordinates = eval_js(
    javascript
)


delivery_target = np.array(

    target_coordinates,

    dtype=float

)


print(
    "\nDelivery target:",
    delivery_target.astype(int)
)



In [ ]:

# =====================================================================
# STEP 12
# ROBOT START POSITION
#
# Lower-left corner
# =====================================================================

robot_start = np.array(

    [

        width * 0.10,

        height * 0.82

    ],

    dtype=float

)


print(
    "\nRobot starting position:",
    robot_start.astype(int)
)


# =====================================================================
# STEP 13
# ASK GEMINI TO PLAN THE TASK
# =====================================================================

if client is not None:

    prompt = f"""

You are the high-level task planner
for a Physical AI service robot.

The robot sees a room using computer vision.

YOLO has detected a real plastic bottle
on a table.

Robot starting position:

{robot_start.astype(int).tolist()}

Detected bottle position:

{bottle_position.astype(int).tolist()}

Human-selected delivery target:

{delivery_target.astype(int).tolist()}

TASK:

The robot must:

1. Navigate from its starting position
   to the detected bottle.

2. Approach the bottle.

3. Pick up the bottle.

4. Carry the bottle to the
   human-selected target.

5. Release the bottle.

6. Stop.

Return a concise numbered high-level
robot task plan.

"""


    response = client.models.generate_content(

        model="gemini-2.5-flash",

        contents=prompt

    )


    print(
        "\n================================="
    )

    print(
        "GEMINI HIGH-LEVEL ROBOT PLAN"
    )

    print(
        "=================================\n"
    )


    print(
        response.text
    )


else:

    print(
        "\nGemini skipped."
    )


# =====================================================================
# STEP 14
# PATH 1
#
# ROBOT START -> REAL BOTTLE
# =====================================================================

FRAMES_TO_BOTTLE = 45


path_to_bottle = []


for t in np.linspace(

    0,

    1,

    FRAMES_TO_BOTTLE

):


    position = (

        robot_start *
        (1 - t)

        +

        bottle_position *
        t

    )


    # Slight curved motion

    position[1] -= (

        25 *

        np.sin(
            np.pi * t
        )

    )


    path_to_bottle.append(

        position.copy()

    )


# =====================================================================
# STEP 15
# PATH 2
#
# REAL BOTTLE -> DELIVERY TARGET
# =====================================================================

FRAMES_TO_TARGET = 55


path_to_target = []


for t in np.linspace(

    0,

    1,

    FRAMES_TO_TARGET

):


    position = (

        bottle_position *
        (1 - t)

        +

        delivery_target *
        t

    )


    position[1] -= (

        35 *

        np.sin(
            np.pi * t
        )

    )


    path_to_target.append(

        position.copy()

    )


In [ ]:
# =====================================================================
# STEP 16
# DRAW ROBOT
# =====================================================================

def draw_robot(

    draw,

    x,

    y,

    carrying=False

):


    # Robot body

    draw.rounded_rectangle(

        [

            x - 30,
            y - 30,

            x + 30,
            y + 32

        ],

        radius=12,

        fill="blue",

        outline="white",

        width=4

    )


    # Robot head

    draw.rounded_rectangle(

        [

            x - 23,
            y - 66,

            x + 23,
            y - 32

        ],

        radius=7,

        fill="lightblue",

        outline="white",

        width=3

    )


    # Eyes

    draw.ellipse(

        [

            x - 14,
            y - 56,

            x - 7,
            y - 49

        ],

        fill="black"

    )


    draw.ellipse(

        [

            x + 7,
            y - 56,

            x + 14,
            y - 49

        ],

        fill="black"

    )


    # Wheels

    draw.ellipse(

        [

            x - 38,
            y + 18,

            x - 20,
            y + 44

        ],

        fill="black"

    )


    draw.ellipse(

        [

            x + 20,
            y + 18,

            x + 38,
            y + 44

        ],

        fill="black"

    )


    # Bottle being carried

    if carrying:

        draw.rounded_rectangle(

            [

                x + 33,
                y - 25,

                x + 47,
                y + 18

            ],

            radius=4,

            fill="lime",

            outline="white",

            width=2

        )


# =====================================================================
# STEP 17
# CREATE ANIMATION
# =====================================================================

frames = []


complete_trajectory = (

    path_to_bottle

    +

    path_to_target

)


for frame_number, position in enumerate(

    complete_trajectory

):


    frame = PILImage.fromarray(

        room.copy()

    )


    draw = ImageDraw.Draw(
        frame
    )


    x = int(
        position[0]
    )


    y = int(
        position[1]
    )


    # =================================================================
    # DRAW TARGET
    # =================================================================

    tx = int(
        delivery_target[0]
    )


    ty = int(
        delivery_target[1]
    )


    draw.ellipse(

        [

            tx - 28,
            ty - 28,

            tx + 28,
            ty + 28

        ],

        outline="red",

        width=7

    )


    draw.text(

        (

            tx + 32,

            ty - 10

        ),

        "DELIVERY TARGET",

        fill="red"

    )


    # =================================================================
    # DETERMINE ROBOT STATE
    # =================================================================

    carrying_bottle = (

        frame_number >=
        FRAMES_TO_BOTTLE

    )


    # =================================================================
    # HIGHLIGHT REAL BOTTLE BEFORE PICKUP
    # =================================================================

    if not carrying_bottle:

        draw.rectangle(

            [

                x1,
                y1,
                x2,
                y2

            ],

            outline="lime",

            width=5

        )


        draw.text(

            (

                x1,

                max(
                    0,
                    y1 - 25
                )

            ),

            "BOTTLE",

            fill="lime"

        )


    # =================================================================
    # DRAW TRAVELLED PATH
    # =================================================================

    if frame_number > 1:

        previous = (

            complete_trajectory[
                :frame_number + 1
            ]

        )


        points = [

            (

                int(p[0]),

                int(p[1])

            )

            for p in previous

        ]


        draw.line(

            points,

            fill="yellow",

            width=5

        )


    # =================================================================
    # DRAW ROBOT
    # =================================================================

    draw_robot(

        draw,

        x,

        y,

        carrying=carrying_bottle

    )


    # =================================================================
    # STATUS MESSAGE
    # =================================================================

    if (
        frame_number <
        FRAMES_TO_BOTTLE - 5
    ):

        status = (

            "PERCEPTION -> NAVIGATION: "
            "Moving to YOLO-detected bottle"

        )


    elif (
        frame_number <
        FRAMES_TO_BOTTLE
    ):

        status = (

            "MANIPULATION: "
            "Picking up bottle"

        )


    elif (
        frame_number <
        len(
            complete_trajectory
        ) - 5
    ):

        status = (

            "NAVIGATION: "
            "Carrying bottle to target"

        )


    else:

        status = (

            "MANIPULATION: "
            "Releasing bottle"

        )


    draw.text(

        (

            20,

            20

        ),

        status,

        fill="white",

        stroke_width=3,

        stroke_fill="black"

    )


    draw.text(

        (

            20,

            52

        ),

        f"Physical AI Step: {frame_number}",

        fill="white",

        stroke_width=2,

        stroke_fill="black"

    )


    frames.append(
        frame
    )


# =====================================================================
# STEP 18
# CREATE FINAL FRAME
#
# Bottle is now at the selected target.
# =====================================================================

final_frame = PILImage.fromarray(

    room.copy()

)


draw = ImageDraw.Draw(
    final_frame
)


tx = int(
    delivery_target[0]
)


ty = int(
    delivery_target[1]
)


# Target

draw.ellipse(

    [

        tx - 30,
        ty - 30,

        tx + 30,
        ty + 30

    ],

    outline="red",

    width=7

)


# Robot

draw_robot(

    draw,

    tx,

    ty,

    carrying=False

)


# Bottle delivered

draw.rounded_rectangle(

    [

        tx + 35,
        ty - 25,

        tx + 50,
        ty + 20

    ],

    radius=4,

    fill="lime",

    outline="white",

    width=3

)


draw.text(

    (

        20,

        20

    ),

    "TASK COMPLETE - REAL BOTTLE DELIVERED TO TARGET",

    fill="yellow",

    stroke_width=3,

    stroke_fill="black"

)


# Hold final frame

for _ in range(15):

    frames.append(

        final_frame.copy()

    )


# =====================================================================
# STEP 19
# SAVE ANIMATION
# =====================================================================

output_file = (

    "/content/"
    "physical_ai_real_bottle_delivery.gif"

)


frames[0].save(

    output_file,

    save_all=True,

    append_images=frames[1:],

    duration=140,

    loop=0

)


print(
    "\n================================="
)

print(
    "PHYSICAL AI SIMULATION READY"
)

print(
    "================================="
)


# =====================================================================
# STEP 20
# DISPLAY ANIMATION
# =====================================================================

display(

    Image(
        filename=output_file
    )

)